# DroneCount sweep — validation & coverage overlay

Companion notebook to `run_dronecount_sweep.py`. Loads every `validation_n??.csv`
that the harness produced and overlays them on a single pair of axes so you can
see how the swarm's tracking quality and coverage scale with fleet size.

## How to use

1. Run the sweep:
   ```
   python3 run_dronecount_sweep.py --start 1 --end 50
   ```
   Each run takes the full sim time of one Roxborough run. The script is
   resumable — re-running it skips any drone count whose `validation_n??.csv`
   already exists.

2. Run the cells below. They read every `validation_n??.csv` under
   `_output/dronecount_sweep/` and produce:
   - **Top panel**: SUMO ground truth (single black line, identical across runs
     by construction) + the swarm's estimate at each drone count (coloured
     from dark = 1 drone to bright = 50 drones).
   - **Bottom panel**: `% edges with a fresh scan` per drone count, same colour
     mapping.
   - **Summary scatter**: per-run MAPE and mean coverage as a function of drone
     count — the single-number scaling curves.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Adjust if you moved the scenario.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()), None); assert ROOT is not None, "Run this notebook from within the WUInity repository."; OUTPUT_DIR = ROOT / "Examples/NFDRS4_Behave/Roxborough/_output"

# The harness writes results into a policy-specific subfolder named
# `dronecount_sweep_<RoutingPolicy>` (e.g. `_AcoStigmergic`, `_Raster`) so different
# policies don't clobber each other. Auto-discover them; set POLICY to pick which one
# to plot in the single-policy cells below. (An old run may have produced a plain
# `dronecount_sweep` folder; that's discovered too.)
candidates = sorted([p for p in OUTPUT_DIR.glob("dronecount_sweep*") if p.is_dir()])
if not candidates:
    raise FileNotFoundError(f"No dronecount_sweep* folders under {OUTPUT_DIR}\n"
                            f"Run run_dronecount_sweep.py first.")
print("Available sweep folders:")
for p in candidates:
    n_runs = len(list(p.glob("validation_n*.csv")))
    print(f"  {p.name:40s}  ({n_runs} runs)")

# Pick one policy for the single-policy cells. Defaults to the first discovered folder;
# change this to e.g. "Raster" or "AcoStigmergic" to select explicitly.
POLICY: str | None = None  # None = use the first discovered folder
if POLICY is not None:
    matches = [p for p in candidates if p.name.endswith(POLICY)]
    if not matches:
        raise FileNotFoundError(f"No sweep folder ending in '{POLICY}'. Have: {[p.name for p in candidates]}")
    SWEEP_DIR = matches[0]
else:
    SWEEP_DIR = candidates[0]
print(f"\nUsing SWEEP_DIR = {SWEEP_DIR.name}")

files = sorted(SWEEP_DIR.glob("validation_n*.csv"),
               key=lambda p: int(p.stem.split("_n")[1]))
if not files:
    raise FileNotFoundError(f"No validation_n*.csv under {SWEEP_DIR}.")

drone_counts = [int(p.stem.split("_n")[1]) for p in files]
print(f"Found {len(files)} sweep runs, drone counts: {min(drone_counts)}..{max(drone_counts)}")
missing = sorted(set(range(min(drone_counts), max(drone_counts) + 1)) - set(drone_counts))
if missing:
    print(f"WARNING: {len(missing)} drone counts missing from the sweep: {missing[:10]}{' ...' if len(missing) > 10 else ''}")


In [ ]:
# Load all runs into a dict keyed by N. Each value is a DataFrame with columns
# sim_time_s, truth, swarm_estimate, coverage_pct.
runs: dict[int, pd.DataFrame] = {}
for fp, n in zip(files, drone_counts):
    runs[n] = pd.read_csv(fp)

# The truth column is identical across runs (SUMO is deterministic w.r.t. drone count).
# Use the first run's truth as the reference.
ref = runs[drone_counts[0]]
print(f"Reference run: N={drone_counts[0]}, {len(ref)} samples, "
      f"time={ref.sim_time_s.min():.0f}..{ref.sim_time_s.max():.0f}s")
print(f"Peak ground truth vehicles: {ref.truth.max():.0f}")

In [ ]:
# Two-panel overlay: validation curves on top, coverage on the bottom.
cmap = plt.get_cmap("viridis")
n_min, n_max = min(drone_counts), max(drone_counts)
norm = plt.Normalize(vmin=n_min, vmax=n_max)

# Mark a few representative counts in the legend rather than all 50.
legend_marks = sorted({n_min, max(n_min, n_max // 4),
                       max(n_min, n_max // 2), max(n_min, (3 * n_max) // 4),
                       n_max})

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

axes[0].plot(ref.sim_time_s, ref.truth, color="black", linewidth=2.2,
             label="SUMO ground truth", zorder=10)

for n, df in runs.items():
    c = cmap(norm(n))
    lbl = f"drones = {n}" if n in legend_marks else None
    axes[0].plot(df.sim_time_s, df.swarm_estimate, color=c, linewidth=0.9,
                 alpha=0.75, label=lbl)
    axes[1].plot(df.sim_time_s, df.coverage_pct, color=c, linewidth=0.9,
                 alpha=0.75, label=lbl)

axes[0].set_ylabel("# vehicles in network")
axes[0].set_title("Validation: total vehicles — swarm estimate vs SUMO ground truth")
axes[0].grid(alpha=0.25)
axes[0].legend(loc="upper left", fontsize=8, framealpha=0.9)

axes[1].set_ylabel("% edges with a fresh scan")
axes[1].set_xlabel("simulation time [s]")
axes[1].grid(alpha=0.25)
axes[1].legend(loc="upper left", fontsize=8, framealpha=0.9)

# Colour-bar to convert hue back to drone count.
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, orientation="vertical", aspect=40, pad=0.02)
cbar.set_label("drone count")

out = SWEEP_DIR / "dronecount_sweep_overlay.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()

In [ ]:
# Scaling curves: per-N headline numbers vs drone count.
rows = []
for n, df in runs.items():
    mask = df.truth > 1.0
    if mask.sum() == 0:
        mae = float("nan"); mape = float("nan")
    else:
        mae  = float(np.mean(np.abs(df.swarm_estimate[mask] - df.truth[mask])))
        mape = float(np.mean(np.abs(df.swarm_estimate[mask] - df.truth[mask]) / df.truth[mask]) * 100)
    rows.append({
        "drones":           n,
        "mae_vehicles":     mae,
        "mape_pct":         mape,
        "mean_coverage_pct": float(df.coverage_pct.mean()),
        "peak_coverage_pct": float(df.coverage_pct.max()),
    })
summary = pd.DataFrame(rows).sort_values("drones").reset_index(drop=True)
summary.to_csv(SWEEP_DIR / "sweep_summary.csv", index=False)
print(f"saved: {SWEEP_DIR / 'sweep_summary.csv'}")
summary

In [ ]:
# How do MAPE, mean coverage, and peak coverage scale with drone count?
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(summary.drones, summary.mape_pct, marker="o", color="darkorange")
axes[0].set_xlabel("drone count")
axes[0].set_ylabel("MAPE (%)")
axes[0].set_title("Tracking error vs fleet size")
axes[0].grid(alpha=0.25)

axes[1].plot(summary.drones, summary.mean_coverage_pct, marker="o",
             color="tab:purple", label="mean coverage")
axes[1].plot(summary.drones, summary.peak_coverage_pct, marker="o",
             color="tab:purple", linestyle="--", alpha=0.6, label="peak coverage")
axes[1].set_xlabel("drone count")
axes[1].set_ylabel("% edges with a fresh scan")
axes[1].set_title("Coverage vs fleet size")
axes[1].grid(alpha=0.25)
axes[1].legend(loc="lower right")

fig.tight_layout()
out = SWEEP_DIR / "dronecount_sweep_scaling.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved: {out}")
plt.show()